# Gaussian Process Regression

Реализуется регрессия на Gaussian Process с нуля на NumPy:
от `RBF kernel` до подбора гиперпараметров по marginal likelihood.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize


from pathlib import Path

def find_repo_root() -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "notebooks").exists() and (candidate / "requirements.txt").exists():
            return candidate
    return current.parent if current.name == "notebooks" else current

REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "data"
FIGURES_DIR = REPO_ROOT / "figures" / "gaussian_process"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)


 
print("Папка для графиков:", FIGURES_DIR)

    np.random.seed(42)

    plt.rcParams['figure.dpi'] = 100
    plt.rcParams['font.size'] = 12

    print("Версия NumPy:", np.__version__)
    print("Базовая настройка готова.")

In [ ]:
# Реализация RBF kernel
# k(x1, x2) = sigma_f^2 * exp(-||x1 - x2||^2 / (2 * l^2))

def rbf_kernel(X1, X2, length_scale, sigma_f):
    """
    Считает матрицу RBF kernel между X1 и X2.

    Параметры
    ----------
    X1 : ndarray of shape (n1, d)
    X2 : ndarray of shape (n2, d)
    length_scale : float
        Controls the smoothness / correlation length (l).
    sigma_f : float
        Signal standard deviation; scales the overall variance.

    Возвращает
    -------
    K : ndarray of shape (n1, n2)
        Матрицу Грама.
    """
    # Быстро считаем квадраты евклидовых расстояний через broadcasting
    # ||x1 - x2||^2 = ||x1||^2 + ||x2||^2 - 2 * x1^T x2
    X1_sq = np.sum(X1 ** 2, axis=1, keepdims=True)   # (n1, 1)
    X2_sq = np.sum(X2 ** 2, axis=1, keepdims=True).T  # (1, n2)
    sq_dists = X1_sq + X2_sq - 2.0 * (X1 @ X2.T)     # (n1, n2)

    # Отсекаем маленькие отрицательные значения из-за численных ошибок
    sq_dists = np.maximum(sq_dists, 0.0)

    K = sigma_f ** 2 * np.exp(-sq_dists / (2.0 * length_scale ** 2))
    return K


# Небольшая проверка
X_demo = np.array([[0.0], [1.0], [2.0]])
K_demo = rbf_kernel(X_demo, X_demo, length_scale=1.0, sigma_f=1.0)
print("Проверочная kernel-матрица (3x3):")
print(np.round(K_demo, 4))
print("Диагональ должна быть равна 1.0 при sigma_f=1:", np.round(np.diag(K_demo), 4))

In [ ]:
# Предсказание GP
# Predictive mean: mu* = K(X*, X) [K(X,X) + noise*I]^{-1} y
# Predictive cov:  C*  = K(X*, X*) - K(X*, X) [K(X,X) + noise*I]^{-1} K(X, X*)

def gp_predict(X_train, y_train, X_test, length_scale, sigma_f, noise):
    """
    Предсказание в Gaussian Process.

    Использует разложение Холецкого и `np.linalg.solve` без явного обращения матрицы
    для численной устойчивости.

    Parameters
    ----------
    X_train : ndarray (n, d)
    y_train : ndarray (n,)
    X_test  : ndarray (m, d)
    length_scale : float
    sigma_f      : float
    noise        : float  -- observation noise standard deviation

    Returns
    -------
    mu_pred   : ndarray (m,)    предиктивное среднее
    var_pred  : ndarray (m,)    предиктивная дисперсия (диагональ ковариации)
    cov_pred  : ndarray (m, m)  полная предиктивная ковариация
    """
    n = len(X_train)
    jitter = 1e-6  # numerical stability jitter

    # Собираем обучающую kernel-матрицу K(X_train, X_train) + noise^2 * I
    K_train = rbf_kernel(X_train, X_train, length_scale, sigma_f)
    K_train += (noise ** 2 + jitter) * np.eye(n)

    # Поперечная kernel-матрица K(X_test, X_train)
    K_star = rbf_kernel(X_test, X_train, length_scale, sigma_f)  # (m, n)

    # Kernel-матрица для тестовых точек K(X_test, X_test)
    K_star_star = rbf_kernel(X_test, X_test, length_scale, sigma_f)  # (m, m)
    K_star_star += jitter * np.eye(len(X_test))

    # Разложение Холецкого для устойчивости: K_train = L @ L.T
    try:
        L = np.linalg.cholesky(K_train)  # lower triangular
    except np.linalg.LinAlgError:
        # Если разложение не сошлось, добавляем ещё jitter
        K_train += 1e-4 * np.eye(n)
        L = np.linalg.cholesky(K_train)

    # Формально можно расписать две треугольные системы,
    # но здесь оставляем `np.linalg.solve`, чтобы код был короче.
    # Явное обращение матрицы не используем.
    alpha = np.linalg.solve(K_train, y_train)  # (n,)

    # Предиктивное среднее
    mu_pred = K_star @ alpha  # (m,)

    # Предиктивная ковариация
    # V = L^{-1} K(X_train, X_test)  via solve
    V = np.linalg.solve(L, K_star.T)  # (n, m)  --  K_star.T is (n, m)
    cov_pred = K_star_star - V.T @ V  # (m, m)

    # Подрезаем маленькие отрицательные дисперсии из-за округления
    var_pred = np.maximum(np.diag(cov_pred), 0.0)  # (m,)

    return mu_pred, var_pred, cov_pred


print("Функция gp_predict готова.")
print("Используем Холецкого и solve; явного обращения матрицы здесь нет.")

In [ ]:
# Marginal likelihood
# log p(y | X, theta) = -0.5 y^T K^{-1} y - 0.5 log|K| - n/2 log(2*pi)

def log_marginal_likelihood(log_params, X_train, y_train):
    """
    Считает отрицательный log marginal likelihood для подбора гиперпараметров GP.

    Параметры передаются в логарифмическом масштабе, чтобы гарантировать положительность:
        log_params = [log(length_scale), log(sigma_f), log(noise)]

    Возвращает
    -------
    neg_lml : float  (отрицательное значение, удобное для `scipy.optimize.minimize`)
    """
    # Возвращаемся из логарифмов к обычным положительным параметрам
    length_scale, sigma_f, noise = np.exp(log_params)

    n = len(X_train)
    jitter = 1e-6

    # Собираем kernel-матрицу с шумом
    K = rbf_kernel(X_train, X_train, length_scale, sigma_f)
    K += (noise ** 2 + jitter) * np.eye(n)

    # Разложение Холецкого: K = L @ L.T
    try:
        L = np.linalg.cholesky(K)
    except np.linalg.LinAlgError:
        return 1e10  # Return large value if not positive definite

    # Член согласования с данными: -0.5 * y^T K^{-1} y
    # Решаем треугольную систему L @ v = y
    v = np.linalg.solve(L, y_train)
    data_fit = -0.5 * np.dot(v, v)

    # Штраф за сложность: -0.5 * log|K| = -sum(log(diag(L)))
    # так как |K| = |L|^2 и |L| = произведение диагонали
    log_det = np.sum(np.log(np.diag(L)))
    complexity = -log_det

    # Константный член: -n/2 * log(2*pi)
    constant = -0.5 * n * np.log(2.0 * np.pi)

    lml = data_fit + complexity + constant

    # Возвращаем минус, потому что `minimize` ищет минимум
    return -lml


print("Функция log_marginal_likelihood готова.")
print("Параметры подаются в логарифмах, чтобы оптимизатор не уходил в отрицательные значения.")
print("Возвращаем минус LML, чтобы `scipy.optimize.minimize` фактически искал максимум.")

## Почему внизу ограничиваем параметр шума

Если дать оптимизатору полную свободу, он слишком охотно тянет `noise` к нулю.
Для маленькой учебной выборки это быстро превращается в переуверенную и численно
хрупкую модель. Поэтому здесь явно ставим нижнюю границу.

In [ ]:
# Генерируем учебные данные
np.random.seed(42)

# Истинная функция: f(x) = sin(x) + 0.3*cos(2x)
def true_function(x):
    return np.sin(x) + 0.3 * np.cos(2 * x)

X_train = np.array([-3, -2, -1, 0, 1, 2, 3]).reshape(-1, 1)
noise_std = 0.2
y_train = (np.sin(X_train.ravel())
           + 0.3 * np.cos(2 * X_train.ravel())
           + noise_std * np.random.randn(len(X_train)))

X_test = np.linspace(-5, 5, 200).reshape(-1, 1)
y_true  = true_function(X_test.ravel())

print("Учебные данные:")
print(f"  X_train shape: {X_train.shape}")
print(f"  y_train shape: {y_train.shape}")
print(f"  X_test  shape: {X_test.shape}")
print(f"  Использованный шум (std): {noise_std}")

# Быстрый просмотр данных
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(X_test.ravel(), y_true, 'k--', label='Истинная функция', linewidth=1.5)
ax.scatter(X_train.ravel(), y_train, color='red', zorder=5, s=60, label='Точки обучения')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Учебные данные')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'training_data.png', dpi=120, bbox_inches='tight')
plt.show()

In [ ]:
# Подбор гиперпараметров по marginal likelihood

# Минимально допустимый шум; не даём оптимизатору схлопнуть его в ноль
NOISE_MIN = 0.01

# Начальные гиперпараметры в логарифмах: [log(length_scale), log(sigma_f), log(noise)]
log_params_init = np.log([1.0, 1.0, 0.1])

print("Начальные гиперпараметры:")
print(f"  length_scale = {np.exp(log_params_init[0]):.4f}")
print(f"  sigma_f      = {np.exp(log_params_init[1]):.4f}")
print(f"  noise        = {np.exp(log_params_init[2]):.4f}")
print(f"  Начальный neg. LML: {log_marginal_likelihood(log_params_init, X_train, y_train):.4f}")
print()

# Ограничения в логарифмическом масштабе:
#   length_scale : (1e-3, 1e3) -- broad range, no strong prior
#   sigma_f      : (1e-3, 1e3) -- broad range
#   noise        : (log(NOISE_MIN), None)  -- enforces noise >= NOISE_MIN
bounds = [
    (np.log(1e-3), np.log(1e3)),          # log(length_scale)
    (np.log(1e-3), np.log(1e3)),          # log(sigma_f)
    (np.log(NOISE_MIN), None),            # log(noise) >= log(0.01)
]

# Запускаем L-BFGS-B
result = minimize(
    log_marginal_likelihood,
    x0=log_params_init,
    args=(X_train, y_train),
    method='L-BFGS-B',
    bounds=bounds,
    options={'maxiter': 1000, 'ftol': 1e-12, 'gtol': 1e-8}
)

# Достаём оптимальные гиперпараметры обратно из логарифмов
opt_length_scale, opt_sigma_f, opt_noise = np.exp(result.x)

print("Результат оптимизации:")
print(f"  Success: {result.success}")
print(f"  Message: {result.message}")
print()
print("Оптимальные гиперпараметры:")
print(f"  length_scale = {opt_length_scale:.4f}  (initial: 1.0)")
print(f"  sigma_f      = {opt_sigma_f:.4f}  (initial: 1.0)")
print(f"  noise        = {opt_noise:.4f}  (initial: 0.1, true: {noise_std}, min allowed: {NOISE_MIN})")
print(f"  Оптимальный neg. LML: {result.fun:.4f}")
print()
print(f"Замечание: noise ограничен снизу значением {NOISE_MIN}, чтобы избежать")
print("             численной нестабильности и переуверенного подбора.")

In [ ]:
# Предсказание и визуализация с подобранными параметрами

# Считаем предсказание с оптимальными гиперпараметрами
mu_pred, var_pred, cov_pred = gp_predict(
    X_train, y_train, X_test,
    length_scale=opt_length_scale,
    sigma_f=opt_sigma_f,
    noise=opt_noise
)
std_pred = np.sqrt(var_pred)  # predictive standard deviation

x_plot = X_test.ravel()

fig, ax = plt.subplots(figsize=(12, 6))

# Полоса 2 sigma: примерно 95% массы для нормального распределения
ax.fill_between(
    x_plot,
    mu_pred - 2 * std_pred,
    mu_pred + 2 * std_pred,
    alpha=0.25,
    color='blue',
    label='Полоса 2 sigma'
)

# Среднее предсказание
ax.plot(x_plot, mu_pred, 'b-', linewidth=2, label='Среднее предсказание GP')

# Истинная функция
ax.plot(x_plot, y_true, 'k--', linewidth=1.5, label='Истинная функция')

# Точки обучения
ax.scatter(X_train.ravel(), y_train,
           color='red', zorder=5, s=80, label='Точки обучения')

ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('y', fontsize=13)
ax.set_title(
    f'Gaussian Process Regression\n'
    f'(l={opt_length_scale:.2f}, sigma_f={opt_sigma_f:.2f}, noise={opt_noise:.3f})',
    fontsize=14
)
ax.legend(fontsize=11)
ax.set_xlim(-5, 5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'gp_regression.png', dpi=120, bbox_inches='tight')
plt.show()

print("Наблюдение: полоса неопределённости узкая рядом с обучающими точками и")
print("расширяется там, где модель уходит в экстраполяцию (x < -3 и x > 3).")

In [ ]:
# График неопределённости
# Смотрим, как меняется std по оси x, и проверяем, что около обучающих точек она мала

fig, ax = plt.subplots(figsize=(12, 5))

ax.plot(x_plot, std_pred, 'b-', linewidth=2, label='Предиктивное std')

# Немного затемняем область под кривой для наглядности
ax.fill_between(x_plot, 0, std_pred, alpha=0.2, color='blue')

# Вертикальные линии показывают положения обучающих точек
for i, x_t in enumerate(X_train.ravel()):
    label = 'Положение обучающих точек' if i == 0 else None
    ax.axvline(x=x_t, color='red', linestyle='--', alpha=0.7, linewidth=1.2, label=label)

ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('Предиктивное стандартное отклонение', fontsize=13)
ax.set_title('Неопределённость GP: стандартное отклонение по оси x', fontsize=14)
ax.legend(fontsize=11)
ax.set_xlim(-5, 5)
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'uncertainty_curve.png', dpi=120, bbox_inches='tight')
plt.show()

# Численная проверка
train_x_vals = X_train.ravel()
# Находим индексы в X_test, ближайшие к обучающим точкам
closest_idxs = [np.argmin(np.abs(x_plot - xt)) for xt in train_x_vals]
print("Предиктивное std вблизи обучающих точек:")
for xt, idx in zip(train_x_vals, closest_idxs):
    print(f"  x = {xt:+.0f}  -->  std = {std_pred[idx]:.5f}")

In [ ]:
# Выборки из апостериорного GP
# Рисуем случайные функции, согласованные с наблюдениями

np.random.seed(0)
n_samples = 3

# Семплируем из многомерного нормального распределения N(mu_pred, cov_pred)
# Добавляем небольшой jitter на диагональ для устойчивости
cov_stable = cov_pred + 1e-8 * np.eye(len(X_test))
posterior_samples = np.random.multivariate_normal(
    mean=mu_pred,
    cov=cov_stable,
    size=n_samples
)  # shape: (n_samples, m)

fig, ax = plt.subplots(figsize=(12, 6))

# Полоса доверия
ax.fill_between(
    x_plot,
    mu_pred - 2 * std_pred,
    mu_pred + 2 * std_pred,
    alpha=0.15,
    color='blue',
    label='Полоса 2 sigma'
)

# Выборки из апостериора
colors = ['darkorange', 'green', 'purple']
for i, sample in enumerate(posterior_samples):
    ax.plot(x_plot, sample, color=colors[i], linewidth=1.2,
            alpha=0.8, linestyle='-', label=f'Выборка из апостериора {i+1}')

# Среднее предсказание
ax.plot(x_plot, mu_pred, 'b-', linewidth=2.5, label='Среднее GP')

# Истинная функция
ax.plot(x_plot, y_true, 'k--', linewidth=1.5, label='Истинная функция')

# Точки обучения
ax.scatter(X_train.ravel(), y_train,
           color='red', zorder=5, s=80, label='Точки обучения')

ax.set_xlabel('x', fontsize=13)
ax.set_ylabel('y', fontsize=13)
ax.set_title('Выборки из апостериорного распределения GP', fontsize=14)
ax.legend(fontsize=10, loc='upper right')
ax.set_xlim(-5, 5)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'posterior_samples.png', dpi=120, bbox_inches='tight')
plt.show()

print("Все выборки из апостериора проходят через обучающие точки или очень близко к ним,")
print("что и должно происходить при корректном условном распределении GP.")

In [ ]:
# Чувствительность к length_scale
# Сравниваем три значения length_scale, остальные параметры фиксируем

length_scales = [0.5, 1.0, 3.0]
descriptions  = [
    'l=0.5: короткий радиус, извилистая кривая, высокая неопределённость',
    'l=1.0: средний радиус, компромиссный вариант',
    'l=3.0: длинный радиус, гладкая и переуверенная кривая'
]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=True)

for ax, ls, desc in zip(axes, length_scales, descriptions):
    # Предсказываем с этим length_scale, sigma_f и noise оставляем оптимальными
    mu, var, _ = gp_predict(
        X_train, y_train, X_test,
        length_scale=ls,
        sigma_f=opt_sigma_f,
        noise=opt_noise
    )
    std = np.sqrt(var)

    # Полоса доверия
    ax.fill_between(
        x_plot, mu - 2 * std, mu + 2 * std,
        alpha=0.25, color='blue', label='Полоса 2 sigma'
    )

    # Среднее предсказание
    ax.plot(x_plot, mu, 'b-', linewidth=2, label='Среднее GP')

    # Истинная функция
    ax.plot(x_plot, y_true, 'k--', linewidth=1.5, label='Истинная функция')

    # Точки обучения
    ax.scatter(X_train.ravel(), y_train,
               color='red', zorder=5, s=70, label='Точки обучения')

    ax.set_title(f'Length Scale l = {ls}\n{desc}', fontsize=11)
    ax.set_xlabel('x', fontsize=12)
    ax.set_xlim(-5, 5)
    ax.set_ylim(-3, 3)

axes[0].set_ylabel('y', fontsize=12)
axes[0].legend(fontsize=9, loc='upper left')

fig.suptitle(
    'GP Regression: как length_scale меняет гладкость и неопределённость',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'length_scale_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

print("Коротко по влиянию length_scale:")
print("  l=0.5 --> Каждая обучающая точка влияет почти только на ближайшую окрестность.")
print("            Вдали от данных предсказание быстро становится рваным и неуверенным.")
print("  l=1.0 --> Компромиссный вариант: умеренная гладкость и вменяемая экстраполяция.")
print("  l=3.0 --> Длинная корреляция по оси x; кривая становится очень гладкой")
print("            и местами слишком уверенной в глобальном тренде.")